# धडा १३ - एजंट मेमरी


## सेटअप

हे नोटबुक **Microsoft Agent Framework** (MAF) वापरून **सतत स्मृती** असलेला ट्रॅव्हल बुकिंग एजंट कसा तयार करायचा हे दर्शविते.

तुम्हाला वेगवेगळ्या प्रकारच्या एजंट स्मृती — कार्यरत, अल्पकालीन, आणि दीर्घकालीन — कशा प्रकारे एजंटची संभाषणे दरम्यान माहिती ठेवण्यात आणि वापरण्यात परिणाम करतात हे शिकता येईल.

**पूर्वअटी:**
- एका Azure AI Foundry प्रकल्पासह एक तैनात चॅट मॉडेल (उदा. `gpt-4o-mini`).
- Azure CLI मध्ये लॉगिन केलेले — तुमच्या टर्मिनलमध्ये `az login` चालवा.
- `AZURE_AI_PROJECT_ENDPOINT` — तुमच्या Azure AI Foundry प्रकल्पाचा एंडपॉइंट.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तैनात मॉडेलचे नाव.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## एजंट मेमरीच्या प्रकार

एआय एजंट विविध प्रकारच्या मेमरीचा उपयोग करू शकतात, ज्याचा प्रत्येकाचा वेगळा उद्देश असतो:

### वर्किंग मेमरी
संवाद थ्रेडचं स्वतः — एका सत्रात देवाणघेवाण केलेले संदेश. एजंट त्या थ्रेडमधील आधीचे संदेश परत पाहू शकतो जेणेकरून सहमती कायम राहील. MAF मध्ये हे **`agent.create_session()`** द्वारे तयार केले जाते, जे `AgentSession` परत करते.

### शॉर्ट-टर्म मेमरी
अशी माहिती जी एखाद्या कार्य किंवा सत्रासाठी टिकून राहते पण कायमची साठवली जात नाही. उदाहरणार्थ, एजंट बहु-टर्न नियोजन संभाषणात तथ्ये जमा करू शकतो आणि त्यांचा वापर अंतिम प्रवास यादी तयार करण्यासाठी करतो.

### लाँग-टर्म मेमरी
अशी प्राधान्ये आणि तथ्ये जी **सत्रांमध्ये सतत टिकून राहतात**. परत येणाऱ्या वापरकर्त्याला त्यांचे आहार प्रतिबंध किंवा प्रवास शैली पुन्हा सांगावे लागणार नाही. लाँग-टर्म मेमरी सहसा बाह्य साठवणुकीने समर्थित होते — एक डेटाबेस, फाइल किंवा व्हेक्टर निर्देशांक — आणि एजंटपर्यंत टूल्सद्वारे पोहोचवली जाते.


## सत्रांसह कार्यरत स्मृती

स्मृतीचा सर्वात साधा प्रकार म्हणजे संभाषण सत्र. जेव्हा तुम्ही एका सत्राला (जे `agent.create_session()` द्वारे तयार केले जाते) सलग `agent.run()` कॉल्समध्ये पास करता, तेव्हा एजंट त्या संभाषणाचा संपूर्ण इतिहास पाहू शकतो आणि आधीच्या तपशीलांना आठवू शकतो.

चला एक प्रवास एजंट तयार करू आणि कार्यरत स्मृतिचे प्रदर्शन करू.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

एजंटने बजेट बरोबर आठवले कारण दोन्ही संदेशांत एकाच सत्राची देवाणघेवाण आहे. हे **कार्यशील स्मृती** आहे — ते फक्त सत्राच्या आयुष्यासाठी अस्तित्वात असते.

### नवीन थ्रेडमध्ये काय होते?

जर आपण **नवीन** सत्र तयार केले, तर एजंटला पूर्वीच्या संभाषणाची कोणतीही आठवण नसतेः


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## दीर्घकालीन स्मृती नमुना

वापरकर्त्याच्या प्राधान्यक्रमांना **सत्रांमध्ये** लक्षात ठेवण्यासाठी, आपल्याला अशी स्थायी संचयस्थळ आवश्यक आहे जी संभाषण थ्रेडच्या बाहेर राहते. एजंट या संचयस्थळाला **tools** च्या माध्यमातून प्रवेश करतो — अशा फंक्शन्स जे माहिती साठवण्यासाठी आणि पुनर्प्राप्त करण्यासाठी कॉल करू शकतात.

खाली आपण एक सोपा इन-मेमरी प्राधान्य संचयस्थळ (उत्पादनात आपण हे डेटाबेस किंवा वेक्टर निर्देशांकाने समर्थित कराल) अमलात आणतो आणि ते एजंट वापरू शकेल अशा साधनांप्रमाणे उघड करतो.

### आर्किटेक्चर
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — प्रथमच येणाऱ्या वापरकर्त्याने वर्धापनदिनाची ट्रिप बुक केली

सारा प्रथमच भेट देते. एजंटने तिच्या प्राधान्यक्रमांना टूल्सच्या माध्यमातून साठवावे आणि त्यांचा वापर करुन हॉटेल्सची शिफारस करावी.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### परिदृश्य 2 — सारा काही आठवडे नंतर परत येते

सारा एक **पूर्णपणे नवीन थ्रेड** सुरू करते (नवीन सत्राचे अनुकरण करत आहे). कार्यशील मेमरी रिकामी आहे, परंतु दीर्घकालीन प्राधान्य संचात तिची माहिती अजूनही आहे. एजंटने ती माहिती पुनःप्राप्त करून शिफारसी वैयक्तिक करण्यासाठी वापरावी.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## सारांश

या धड्यात आपण तिघ प्रकारच्या एजंट मेमरीबद्दल आणि त्यांना Microsoft Agent Framework सह कसे राबवायचे ते शिकलात:

| मेमरीचा प्रकार | MAF यंत्रणा | आयुष्यकाल |
|---|---|---|
| **वर्किंग** | `agent.create_session()` | एकच संभाषण |
| **शॉर्ट-टर्म** | थ्रेडमधील जमा केलेला संदर्भ | एकच कार्य / सत्र |
| **लाँग-टर्म** | `@tool` फंक्शन्सद्वारे प्रवेश केलेले बाह्य संच | सत्रांमध्ये |

### मुख्य मुद्दे
1. **`agent.create_session()`** वर्किंग मेमरी देते — एजंटसत्रामध्ये पूर्ण संभाषण इतिहास पाहू शकतो.
2. **नवीन सत्रे संदर्भ गमावतात** — लाँग-टर्म मेमरीशिवाय एजंट मागील संभाषणे आठवू शकत नाही.
3. **`@tool` फंक्शन्स** हा मधोमधील सेतू आहेत — ते एजंटला एक सातत्यपूर्ण संचातून माहिती जतन करण्याची आणि पुनर्प्राप्त करण्याची परवानगी देतात.
4. **वैयक्तिकरण वेळेनुसार सुधारते** — जितक्या जास्त प्राधान्यांची नोंद होईल तितकी एजंटची शिफारस चांगली होईल.

### प्रत्यक्ष उपयोग
- **कस्टमर सर्व्हिस**: ग्राहकांचा इतिहास आणि प्राधान्ये लक्षात ठेवणे
- **वैयक्तिक सहाय्यक**: दिवस किंवा आठवड्यांदरम्यान संदर्भ राखणे
- **आरोग्य सेवा**: रुग्णांची माहिती आणि प्राधान्ये ट्रॅक करणे
- **ई-कॉमर्स**: इतिहासावर आधारित वैयक्तिकृत खरेदी

### पुढील पावले
- इन-मेमरी डिक्ट बदलून डेटाबेस किंवा वेक्टर स्टोअर वापरणे (उदा. Azure AI Search)
- वेळेवर माहितीकरिता मेमरीची कालमर्यादा वाढवणे
- सामायिक मेमरीसह मल्टि-एजंट सिस्टम तयार करणे
- ज्ञान-ग्राफ समर्थीत मेमरीसाठी Cognee नोटबुक शोधणे


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**सूचना**:  
हा दस्तऐवज AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) वापरून अनुवादित केला आहे. आम्ही अचूकतेसाठी प्रयत्नशील असलो तरी, कृपया ध्यानात ठेवा की स्वयंचलित अनुवादांमध्ये त्रुटी किंवा अचूकतेत कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहिती साठी व्यावसायिक मानवी अनुवादशील सल्ला घेणे शिफारसीय आहे. या अनुवादाच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमजुती किंवा चुकीच्या अर्थ लावण्याची जबाबदारी आम्ही स्वीकारत नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
